In [1]:
import torch, ultralytics, cv2
print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))
print("ultralytics", ultralytics.__version__)
print("opencv", cv2.__version__)

2.11.0+cu128 True NVIDIA GeForce RTX 5070 Laptop GPU
ultralytics 8.4.173
opencv 5.0.0


## <h3 style ="Color:red ;"> <Center>Cell 2: Check your videos  </Center></h3>

In [2]:
import cv2
from pathlib import Path

for v in sorted(Path("clips").glob("*.mp4")):
    cap = cv2.VideoCapture(str(v))
    fps = cap.get(cv2.CAP_PROP_FPS)
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    w, h = int(cap.get(3)), int(cap.get(4))
    print(f"{v.name}: {w}x{h}, {fps:.0f} fps, {n} frames, {n/fps:.0f} sec")
    cap.release()

AirPods_1.mp4: 848x480, 30 fps, 394 frames, 13 sec
AirPods_2.mp4: 848x480, 30 fps, 937 frames, 32 sec
AirPods_3.mp4: 848x480, 30 fps, 2391 frames, 81 sec
AirPods_4.mp4: 480x864, 29 fps, 1572 frames, 53 sec


Cell 3: Extract frames


In [3]:
import cv2
from pathlib import Path

N_PER_VIDEO = 60
OUT = Path("frames")

for v in sorted(Path("clips").glob("*.mp4")):
    out_dir = OUT / v.stem
    out_dir.mkdir(parents=True, exist_ok=True)
    cap = cv2.VideoCapture(str(v))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    step = max(total // N_PER_VIDEO, 1)
    saved = 0
    
    for idx in range(0, total, step):
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ok, frame = cap.read()
        if not ok:
            continue
        cv2.imwrite(str(out_dir / f"{v.stem}_{idx:05d}.jpg"), frame)
        saved += 1
    cap.release()
    print(f"{v.stem}: {saved} frames saved")

AirPods_1: 66 frames saved
AirPods_2: 63 frames saved
AirPods_3: 62 frames saved
AirPods_4: 61 frames saved


Cell 4: Check the result

In [4]:
for d in sorted(Path("frames").iterdir()):
    print(d.name, len(list(d.glob("*.jpg"))))

AirPods_1 66
AirPods_2 63
AirPods_3 62
AirPods_4 61


In [5]:
from pathlib import Path
import re, collections

imgs = list(Path("export_all").rglob("*.jpg"))
print("total images:", len(imgs))

per_video = collections.Counter()
with_box = collections.Counter()
for p in imgs:
    m = re.match(r"(AirPods_\d+)", p.name)
    v = m.group(1) if m else "UNKNOWN"
    per_video[v] += 1
    lbl = p.parent.parent / "labels" / (p.stem + ".txt")
    if lbl.exists() and lbl.read_text().strip():
        with_box[v] += 1

for v in sorted(per_video):
    print(f"{v}: {per_video[v]} images, {with_box[v]} with boxes")

print(open("export_all/data.yaml").read())

total images: 252
AirPods_1: 66 images, 66 with boxes
AirPods_2: 63 images, 63 with boxes
AirPods_3: 62 images, 62 with boxes
AirPods_4: 61 images, 61 with boxes
train: ../train/images
val: ../valid/images
test: ../test/images

nc: 1
names: ['AirPods']

roboflow:
  workspace: hdmodel
  project: airpods_tracking_v2
  version: 1
  license: CC BY 4.0
  url: https://universe.roboflow.com/hdmodel/airpods_tracking_v2/dataset/1


Cell 1: Split by video

In [6]:
from pathlib import Path
import shutil, re

SRC = Path("export_all")
OUT = Path("dataset")
shutil.rmtree(OUT, ignore_errors=True)

split_of = {"AirPods_1": "train", "AirPods_2": "train",
            "AirPods_3": "val",   "AirPods_4": "test"}

for img in SRC.rglob("*.jpg"):
    m = re.match(r"(AirPods_\d+)", img.name)
    if not m:
        continue
    split = split_of[m.group(1)]
    lbl = img.parent.parent / "labels" / (img.stem + ".txt")
    (OUT / split / "images").mkdir(parents=True, exist_ok=True)
    (OUT / split / "labels").mkdir(parents=True, exist_ok=True)
    shutil.copy(img, OUT / split / "images" / img.name)
    shutil.copy(lbl, OUT / split / "labels" / lbl.name)

for s in ["train", "val", "test"]:
    print(s, len(list((OUT / s / "images").glob("*.jpg"))))

train 129
val 62
test 61


Cell 2: Create data.yaml

In [7]:
import yaml
cfg = {
    "path": Path("dataset").resolve().as_posix(),
    "train": "train/images",
    "val": "val/images",
    "test": "test/images",
    "names": {0: "AirPods"},
}
yaml.safe_dump(cfg, open("dataset/data.yaml", "w"))
print(open("dataset/data.yaml").read())

names:
  0: AirPods
path: S:/MODEL/Object/dataset
test: test/images
train: train/images
val: val/images



Cell 3: Train

In [8]:
from ultralytics import YOLO

model = YOLO("yolo11s.pt")
model.train(
    data="dataset/data.yaml",
    epochs=60,
    imgsz=640,
    batch=16,
    device=0,
    workers=0,
    patience=15,
    project="airpods_runs",
    name="v1",
    exist_ok=True,
)

New https://pypi.org/project/ultralytics/8.4.199 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.173  Python-3.11.9 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5070 Laptop GPU, 8151MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=60, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x00000202E7130790>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0480

In [9]:
from pathlib import Path

bad = []
for f in Path("dataset").rglob("labels/*.txt"):
    new = []
    for ln in f.read_text().strip().splitlines():
        p = ln.split()
        if len(p) != 5:                      # polygon -> convert to box
            xs = list(map(float, p[1::2])); ys = list(map(float, p[2::2]))
            x1, x2, y1, y2 = min(xs), max(xs), min(ys), max(ys)
            ln = f"{p[0]} {(x1+x2)/2:.6f} {(y1+y2)/2:.6f} {x2-x1:.6f} {y2-y1:.6f}"
            bad.append(f.name)
        new.append(ln)
    f.write_text("\n".join(new) + "\n")

for c in Path("dataset").rglob("labels.cache"):
    c.unlink()                               # force a fresh scan
print("fixed files:", bad)

fixed files: ['AirPods_1_00156_jpg.rf.4b252e15b055d839a896c4a901f71605.txt']


In [10]:
from ultralytics import YOLO
from pathlib import Path

model = YOLO("yolo11s.pt")
model.train(
    data="dataset/data.yaml",
    epochs=80,
    imgsz=640,
    batch=16,
    device=0,
    workers=0,
    amp=False,            # avoids mixed-precision instability
    optimizer="SGD",      # steadier than auto AdamW on tiny data
    lr0=0.005,
    warmup_epochs=5,
    patience=40,          # don't stop early
    project=str(Path("airpods_runs").resolve()),
    name="v2",
    exist_ok=True,
)

New https://pypi.org/project/ultralytics/8.4.199 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.173  Python-3.11.9 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5070 Laptop GPU, 8151MiB)
engine\trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=80, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.005, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x00000200803B2490>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0480

Cell 1: Honest test on video 4

In [11]:
from ultralytics import YOLO
best = YOLO("airpods_runs/v2/weights/best.pt")
m = best.val(data="dataset/data.yaml", split="test", workers=0, device=0, verbose=False)
print("TEST mAP50:", round(m.box.map50, 3), "mAP50-95:", round(m.box.map, 3),
      "P:", round(m.box.mp, 3), "R:", round(m.box.mr, 3))

Ultralytics 8.4.173  Python-3.11.9 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5070 Laptop GPU, 8151MiB)
YOLO11s summary (fused): 100 layers, 9,413,187 parameters, 0 gradients, 21.4 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 71.531.3 MB/s, size: 28.3 KB)
val: Scanning S:\MODEL\Object\dataset\test\labels... 61 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 61/61 1.0Kit/s 0.1s
val: New cache created: S:\MODEL\Object\dataset\test\labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 2.0it/s 2.0s0.8s
                   all         61         61      0.919      0.918      0.957        0.9
Speed: 1.6ms preprocess, 11.5ms inference, 0.0ms loss, 3.8ms postprocess per image
Results saved to S:\MODEL\Object\runs\detect\val
TEST mAP50: 0.957 mAP50-95: 0.9 P: 0.919 R: 0.918


Cell 2: Track on video 4

In [12]:
r = best.track(source="clips/AirPods_4.mp4", tracker="bytetrack.yaml",
               save=True, device=0, conf=0.4, verbose=False, project="airpods_runs", name="track_v4", exist_ok=True)

requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   -------------- ------------------------- 0.5/1.5 MB 2.1 MB/s eta 0:00:01
   -------------- ------------------------- 0.5/1.5 MB 2.1 MB/s eta 0:00:01
   --------------------- ------------------ 0.8/1.5 MB 1.2 MB/s eta 0:00:01
   --------------------- ------------------ 0.8/1.5 MB 1.2 MB/s eta 0:00:01
   ---------------------------- ----------- 1.0/1.5 MB 838.4 kB/s eta 0:00:01
   ----------------------------------- ---- 1.3/1.5 MB 919.0 kB/s eta 0:00:01
   ----------------------------------- ---- 1.3/1.5 MB 919.0 kB/s eta 0:00:01
   ----------------------------------- ---- 1.3/1.5 MB 919.0 kB/s eta 0:00:01
   ---------------------------------------- 1.5/1.5 MB 748.3 kB/s  0:00:02

[notice] A 

In [13]:
from ultralytics import YOLO
best = YOLO("airpods_runs/v2/weights/best.pt")   # or runs/detect/airpods_runs/v2/weights/best.pt

ids, frames, detected = set(), 0, 0
for r in best.track(source="clips/AirPods_4.mp4", tracker="bytetrack.yaml",
                    conf=0.4, device=0, stream=True, verbose=False):
    frames += 1
    if r.boxes.id is not None:
        detected += 1
        ids.update(r.boxes.id.int().tolist())

print("frames:", frames, "| frames with a track:", detected,
      f"({detected/frames:.0%})")
print("unique IDs:", len(ids), sorted(ids))

frames: 1572 | frames with a track: 1572 (100%)
unique IDs: 52 [1, 2, 4, 7, 10, 15, 17, 19, 22, 24, 27, 28, 34, 35, 42, 43, 44, 47, 48, 49, 51, 57, 61, 62, 65, 68, 69, 71, 77, 84, 85, 87, 88, 90, 91, 93, 94, 98, 100, 101, 102, 103, 104, 106, 108, 111, 112, 116, 117, 119, 124, 125]


In [14]:
from ultralytics import YOLO
import collections

best = YOLO("airpods_runs/v2/weights/best.pt")
boxes_per_frame = collections.Counter()
id_life = collections.Counter()

for r in best.track(source="clips/AirPods_4.mp4", tracker="bytetrack.yaml",
                    conf=0.4, device=0, stream=True, verbose=False):
    boxes_per_frame[len(r.boxes)] += 1
    if r.boxes.id is not None:
        for i in r.boxes.id.int().tolist():
            id_life[i] += 1

print("boxes per frame (count: frames):", dict(boxes_per_frame))
print("longest-lived IDs:", id_life.most_common(5))

boxes per frame (count: frames): {2: 375, 3: 109, 4: 65, 5: 37, 6: 14, 1: 966, 7: 5, 8: 1}
longest-lived IDs: [(1, 1572), (85, 135), (84, 95), (24, 76), (51, 66)]


In [15]:
ids = set()
for r in best.track(source="clips/AirPods_4.mp4", tracker="botsort.yaml",
                    conf=0.5, device=0, stream=True, verbose=False):
    if r.boxes.id is not None:
        ids.update(r.boxes.id.int().tolist())
print("unique IDs with BoT-SORT:", len(ids))

unique IDs with BoT-SORT: 35
